<a href="https://colab.research.google.com/github/Benxperia/MRes/blob/main/TVB_Hydro_Multimodal_RS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hydrological flow analysis into lakes — Tanana Valley Basin, Alaska.

Datasets included:
  - Landsat time series composites (1978–2024)
  - Sentinel-1 SAR — surface water & inundation (2017–2024)
  - Sentinel-2 Optical — water indices & inundation extent (2017–2024)
  - GEDI LiDAR — ground elevation for flow routing (2019–2024)
  - ArcticDEM + SRTM — hydrological terrain derivatives
      (slope, aspect, TPI, flow accumulation, drainage basins)
  - JRC Global Surface Water — long-term lake extent & seasonality (1984–2024)
  - MODIS — snowmelt timing & surface water dynamics (2000–2024)

# Set up

In [ ]:
from google.colab import drive
drive.mount("/content/drive/", force_remount=True)

import ee
import time
import geopandas as gpd

ee.Authenticate()
ee.Initialize(project='rs-s2-450716')

Mounted at /content/drive/


# Load AOI shapefile

In [ ]:
shapefile_path = "/content/drive/MyDrive/MRes/Landcover analysis/shp/TRVB_upstream_lvl6_dissolve_repro.shp"
gdf = gpd.read_file(shapefile_path)

if gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
    print(f"Reprojected to EPSG:4326")

geom   = gdf.geometry.iloc[0]
coords = list(geom.exterior.coords)
AOI    = ee.Geometry.Polygon(coords)

bounds = gdf.total_bounds
print(f"AOI bounds: {bounds}")
print(f"Center: {(bounds[1]+bounds[3])/2:.4f}, {(bounds[0]+bounds[2])/2:.4f}")

Reprojected to EPSG:4326
AOI bounds: [-147.68755569   61.70831914 -140.36662622   65.12147741]
Center: 63.4149, -144.0271


# Parameters

In [ ]:
ImageName = "Alaska_TRV_Basin"
Cloud_Max = 20

# Temporal windows
LANDSAT_START = 2018
LANDSAT_END   = 2024

SAR_START     = 2022   # Sentinel-1 focus window
SAR_END       = 2024

S2_START      = 2022   # Sentinel-2 focus window
S2_END        = 2024

GEDI_START    = 2019   # GEDI availability
GEDI_END      = 2024

MODIS_START   = 2000
MODIS_END     = 2024

JRC_START     = 2018   # JRC Global Surface Water start
JRC_END       = 2024

# Seasonal windows — split for hydrology:
#   Snowmelt / freshet  : May–June
#   Peak open water     : July–September
FRESHET_START  = 5   # May
FRESHET_END    = 6   # June
SUMMER_START   = 7   # July
SUMMER_END     = 9   # September

print(f"\nProcessing: {ImageName}")
print(f"Landsat: {LANDSAT_START}–{LANDSAT_END} | SAR: {SAR_START}–{SAR_END} | "
      f"S2: {S2_START}–{S2_END} | GEDI: {GEDI_START}–{GEDI_END}")


Processing: Alaska_TRV_Basin
Landsat: 2018–2024 | SAR: 2022–2024 | S2: 2022–2024 | GEDI: 2019–2024


# Shared Utilities

In [ ]:
def exportImage(image, filename, scale=30, folder='Alaska_TRV_Hydro'):
    """Gap-fill with progressive focal mean then export to Google Drive."""
    if image is None:
        print(f"  [SKIP] No data for {filename}")
        return False
    try:
        for radius, iterations in [(1, 1), (2, 2), (3, 3)]:
            kernel = ee.Kernel.circle(radius=radius)
            fill   = image.focal_mean(kernel=kernel, iterations=iterations)
            image  = image.unmask(fill)

        image = image.toFloat()
        task  = ee.batch.Export.image.toDrive(
            image          = image,
            description    = filename,
            folder         = folder,
            fileNamePrefix = filename,
            region         = AOI,
            scale          = scale,
            maxPixels      = 1e13,
            fileFormat     = 'GeoTIFF'
        )
        task.start()
        print(f"  [EXPORT STARTED] {filename}  (scale={scale}m, folder={folder})")
        return True
    except Exception as e:
        print(f"  [ERROR] {filename}: {e}")
        return False


def makeDateRange(year, start_month, end_month):
    return (
        ee.Date.fromYMD(year, start_month, 1),
        ee.Date.fromYMD(year, end_month,   28 if end_month == 2 else 30)
    )

# Landsat

(unchanged from original — preserves full time series)

In [ ]:
def customCloudMask(image):
    qa     = image.select('QA_PIXEL')
    cloud  = qa.bitwiseAnd(1 << 3).neq(0)
    shadow = qa.bitwiseAnd(1 << 4).neq(0)
    return image.updateMask(cloud.Or(shadow).Not())


def hasImagesForYear(collection_id, year,
                     start_month=SUMMER_START, end_month=SUMMER_END):
    sd  = ee.Date.fromYMD(year, start_month, 1)
    ed  = ee.Date.fromYMD(year, end_month,   30)
    col = (ee.ImageCollection(collection_id)
             .filterDate(sd, ed)
             .filterBounds(AOI)
             .filter(ee.Filter.lt('CLOUD_COVER', Cloud_Max)))
    return col.size().getInfo() > 0


def processMSS(year, cloud_max, startDate, endDate):
    col      = ee.ImageCollection('LANDSAT/LM03/C02/T2')
    filtered = (col.filterDate(startDate, endDate).filterBounds(AOI)
                   .filter(ee.Filter.lt('CLOUD_COVER', cloud_max)))
    count    = filtered.size().getInfo()
    if count == 0:
        print(f"  No MSS images for {year}"); return None
    print(f"  {count} MSS images for {year}")
    med  = filtered.median()
    sel  = ee.Image.cat([med.select('B7'), med.select('B6'), med.select('B4'),
                         med.select('B7'), med.select('B6'), med.select('B4'),
                         med.select('B7')]).rename(['B1','B2','B3','B4','B5','B6','B7'])
    mx   = filtered.max()
    mxf  = ee.Image.cat([mx.select('B7'), mx.select('B6'), mx.select('B4'),
                         mx.select('B7'), mx.select('B6'), mx.select('B4'),
                         mx.select('B7')]).rename(['B1','B2','B3','B4','B5','B6','B7'])
    return sel.unmask(mxf).set('year', year)


def processLandsat4to7(year, collection_id, cloud_max, startDate, endDate):
    col      = ee.ImageCollection(collection_id)
    filtered = (col.filterDate(startDate, endDate).filterBounds(AOI)
                   .filter(ee.Filter.lt('CLOUD_COVER', cloud_max)))
    count    = filtered.size().getInfo()
    if count == 0:
        print(f"  No images in {collection_id} for {year}"); return None
    print(f"  {count} images in {collection_id} for {year}")
    masked   = filtered.map(customCloudMask)
    comp     = masked.median()
    sel      = comp.select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'])
    b4_dup   = comp.select('SR_B4')
    selected = ee.Image.cat([sel, b4_dup]).rename(['B1','B2','B3','B4','B5','B6','B7'])
    mx_sel   = masked.max().select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'])
    mx_b4    = masked.max().select('SR_B4')
    mx_fill  = ee.Image.cat([mx_sel, mx_b4]).rename(['B1','B2','B3','B4','B5','B6','B7'])
    return selected.unmask(mx_fill).set('year', year)


def processLandsat7SLCOff(year, collection_id, cloud_max, startDate, endDate):
    col      = ee.ImageCollection(collection_id)
    filtered = (col.filterDate(startDate, endDate).filterBounds(AOI)
                   .filter(ee.Filter.lt('CLOUD_COVER', cloud_max)))
    count    = filtered.size().getInfo()
    if count == 0:
        print(f"  No Landsat 7 SLC-off images for {year}"); return None
    print(f"  {count} Landsat 7 SLC-off images for {year}")
    masked = filtered.map(customCloudMask)
    comp   = masked.median().unmask(masked.max())
    kernel = ee.Kernel.circle(radius=3)
    comp   = comp.unmask(comp.focal_mean(kernel=kernel, iterations=3))
    sel    = comp.select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'])
    b4_dup = comp.select('SR_B4')
    return ee.Image.cat([sel, b4_dup]).rename(['B1','B2','B3','B4','B5','B6','B7']).set('year', year)


def processLandsat8to9(year, collection_id, cloud_max, startDate, endDate):
    col      = ee.ImageCollection(collection_id)
    filtered = (col.filterDate(startDate, endDate).filterBounds(AOI)
                   .filter(ee.Filter.lt('CLOUD_COVER', cloud_max)))
    count    = filtered.size().getInfo()
    if count == 0:
        print(f"  No images in {collection_id} for {year}"); return None
    print(f"  {count} images in {collection_id} for {year}")
    masked    = filtered.map(customCloudMask)
    comp      = masked.median()
    bands_in  = ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']
    bands_out = ['B1','B2','B3','B4','B5','B6','B7']
    selected  = comp.select(bands_in, bands_out)
    mx_fill   = masked.max().select(bands_in, bands_out)
    return selected.unmask(mx_fill).set('year', year)


def findBestLandsatCollection(year):
    if year >= 2021:
        for col in ['LANDSAT/LC09/C02/T1_L2', 'LANDSAT/LC08/C02/T1_L2']:
            if hasImagesForYear(col, year): return col
    elif year >= 2013:
        if hasImagesForYear('LANDSAT/LC08/C02/T1_L2', year): return 'LANDSAT/LC08/C02/T1_L2'
    elif 1999 <= year <= 2012:
        for col in ['LANDSAT/LT05/C02/T1_L2', 'LANDSAT/LE07/C02/T1_L2']:
            if hasImagesForYear(col, year): return col
    elif 1984 <= year <= 1998:
        for col in ['LANDSAT/LT05/C02/T1_L2', 'LANDSAT/LT04/C02/T1_L2']:
            if hasImagesForYear(col, year): return col
    elif 1972 <= year <= 1983:
        if hasImagesForYear('LANDSAT/LM03/C02/T2', year): return 'LANDSAT/LM03/C02/T2'
    return None

# Sentinel-1 SAR

Surface water & Inundation (2017-2024)

Water bodies appear as very low backscatter in VV (calm specular reflection away from sensor). We compute per-year composites for TWO seasonal windows:

* (a) Freshet (May–Jun): captures peak snowmelt inundation

* (b) Summer  (Jul–Sep): captures stable open water extent

Outputs per window:

* VV_dB, VH_dB       — backscatter (specular low = water)

* water_mask          — VV < threshold (-15 dB) binary water proxy

* VV_VH_ratio         — sensitive to surface roughness / flooding

* RVI                 — Radar Vegetation Index (flood under canopy proxy)

In [ ]:
SAR_WATER_THRESHOLD_DB = -15   # VV dB below which pixel is flagged as water
                                # typical open calm water ≈ -20 to -15 dB in Alaska

def processSentinel1_Hydro(year, startDate, endDate, season_label):
    col = (ee.ImageCollection('COPERNICUS/S1_GRD')
             .filterBounds(AOI)
             .filterDate(startDate, endDate)
             .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
             .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
             .filter(ee.Filter.eq('instrumentMode', 'IW'))
             .select(['VV', 'VH']))

    count = col.size().getInfo()
    if count == 0:
        print(f"  No Sentinel-1 images for {year} ({season_label})")
        return None
    print(f"  {count} Sentinel-1 scenes for {year} ({season_label})")

    # Use minimum VV composite for inundation — flooded pixels
    # maintain low backscatter throughout the season window
    min_comp    = col.select('VV').min()   # lowest backscatter = most water-like
    median_comp = col.median()

    vv = median_comp.select('VV').rename('VV_dB')
    vh = median_comp.select('VH').rename('VH_dB')

    # Binary water proxy: VV minimum below threshold
    water_mask = min_comp.lt(SAR_WATER_THRESHOLD_DB).rename('SAR_water_mask')

    # Linear-space ratio and RVI
    vv_lin   = ee.Image(10).pow(vv.divide(10))
    vh_lin   = ee.Image(10).pow(vh.divide(10))
    ratio    = vv_lin.divide(vh_lin).log10().multiply(10).rename('VV_VH_ratio_dB')
    rvi      = vh_lin.multiply(4).divide(vv_lin.add(vh_lin)).rename('RVI')

    # Inundation frequency: fraction of scenes where VV < threshold
    inund_freq = (col.select('VV')
                     .map(lambda img: img.lt(SAR_WATER_THRESHOLD_DB).rename('inundated'))
                     .mean()
                     .rename('inundation_frequency'))

    return (ee.Image.cat([vv, vh, water_mask, ratio, rvi, inund_freq])
              .set('year', year)
              .set('season', season_label))

# Sentinel-2

Water indicies & inundation (2017-2024)

Key indices for hydrology:
* NDWI  = (Green - NIR)  / (Green + NIR)     — open water (McFeeters 1996)
* MNDWI = (Green - SWIR1)/ (Green + SWIR1)   — better suppression of built-up
* AWEI  = 4*(Green-SWIR1) - (0.25*NIR+2.75*SWIR2)  — automated water extraction
* AWEInsh = Blue + 2.5*Green - 1.5*(NIR+SWIR1) - 0.25*SWIR2  — no-shadow variant
* NDSI  = (Green - SWIR1)/ (Green + SWIR1)   — snow / ice (same formula as MNDWI; use in freshet window to separate snow from water)

Also outputs: water binary mask (MNDWI > 0)

In [ ]:
def maskS2clouds(image):
    qa         = image.select('QA60')
    cloud_mask = (qa.bitwiseAnd(1 << 10).eq(0)
                    .And(qa.bitwiseAnd(1 << 11).eq(0)))
    return image.updateMask(cloud_mask).divide(10000)


def processSentinel2_Hydro(year, startDate, endDate, season_label):
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
             .filterBounds(AOI)
             .filterDate(startDate, endDate)
             .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', Cloud_Max))
             .map(maskS2clouds))

    count = col.size().getInfo()
    if count == 0:
        print(f"  No Sentinel-2 images for {year} ({season_label})")
        return None
    print(f"  {count} Sentinel-2 scenes for {year} ({season_label})")

    comp  = col.median()
    blue  = comp.select('B2')
    green = comp.select('B3')
    red   = comp.select('B4')
    nir   = comp.select('B8')
    swir1 = comp.select('B11')
    swir2 = comp.select('B12')

    # Base spectral bands (10 m)
    bands    = comp.select(['B2','B3','B4','B8','B11','B12'],
                           ['Blue','Green','Red','NIR','SWIR1','SWIR2'])

    # Water / snow indices
    ndwi  = comp.normalizedDifference(['B3','B8']).rename('NDWI')
    mndwi = comp.normalizedDifference(['B3','B11']).rename('MNDWI')
    ndsi  = comp.normalizedDifference(['B3','B11']).rename('NDSI')   # same bands; interpret by season

    # AWEI (shadow-inclusive variant)
    awei  = (green.multiply(4)
                  .subtract(swir1)
                  .subtract(nir.multiply(0.25).add(swir2.multiply(2.75)))
                  .rename('AWEI'))

    # AWEI no-shadow variant
    awei_nsh = (blue.add(green.multiply(2.5))
                    .subtract(nir.add(swir1).multiply(1.5))
                    .subtract(swir2.multiply(0.25))
                    .rename('AWEInsh'))

    # Binary water mask: MNDWI > 0 (positive = water more likely than vegetation)
    water_mask = mndwi.gt(0).rename('S2_water_mask')

    # Water frequency across the season: fraction of scenes where MNDWI > 0
    water_freq = (col.map(lambda img: img.normalizedDifference(['B3','B11'])
                                         .gt(0)
                                         .rename('water'))
                     .mean()
                     .rename('water_frequency'))

    return (ee.Image.cat([bands, ndwi, mndwi, ndsi, awei, awei_nsh,
                          water_mask, water_freq])
              .set('year', year)
              .set('season', season_label))

# GEDI

Fround elevation for flow routing (2019-2024)

For hydrology we want ground shots, not canopy tops. We select shots where beam sensitivity is high and the ground-surface elevation is well-defined.
Outputs: ground elevation (elev_lowestmode), RH98 (canopy height), beam sensitivity, shot quality.

In [ ]:
def processGEDI_Hydro(year, startDate, endDate):
    """
    GEDI L2A ground elevation and canopy height.

    NOTE ON ALASKA COVERAGE: GEDI is mounted on the ISS which orbits
    up to ~51.6 degrees inclination. The Tanana Valley (~65 N) sits outside
    GEDI primary coverage, so data is very sparse or absent in most years.
    This function addresses this by:
      1. Falling back to the full-year window if the summer window is empty
      2. Relaxing the sensitivity threshold from 0.95 to 0.90
    If no data is found for any window, the year is skipped cleanly.
    This is expected behaviour for high latitudes.
    """
    sd_full = ee.Date.fromYMD(year, 1, 1)
    ed_full = ee.Date.fromYMD(year, 12, 31)

    def filterGEDI(sd, ed, label):
        col = (ee.ImageCollection('LARSE/GEDI/GEDI02_A_002_MONTHLY')
                 .filterBounds(AOI)
                 .filterDate(sd, ed)
                 .map(lambda img: img.updateMask(
                       img.select('quality_flag').eq(1)
                         .And(img.select('degrade_flag').eq(0))
                         .And(img.select('sensitivity').gte(0.90))
                 )))
        count = col.size().getInfo()
        print(f"  GEDI L2A {label}: {count} monthly mosaics found")
        return col, count

    # Try summer window first, fall back to full year
    gedi_2a, count = filterGEDI(startDate, endDate, 'summer window')
    if count == 0:
        print(f"  Trying full-year window for {year}...")
        gedi_2a, count = filterGEDI(sd_full, ed_full, 'full year')

    if count == 0:
        print(f"  No GEDI data found for {year} -- AOI likely outside ISS orbit coverage")
        return None

    ground_elev  = gedi_2a.select('elev_lowestmode').mean().rename('ground_elevation_m')
    elev_quality = gedi_2a.select('sensitivity').mean().rename('beam_sensitivity')
    rh98         = gedi_2a.select('rh98').mean().rename('canopy_height_rh98_m')

    return (ee.Image.cat([ground_elev, elev_quality, rh98])
              .set('year', year))

# Hydrological terrain derivatives

(static, export once)

Uses MERIT Hydro — a hydrologically conditioned DEM purpose-built for flow routing (void-filled, depression-breached, river-burned).
Also exports ArcticDEM and SRTM derivatives for comparison.

MERIT Hydro outputs:
* elv   — elevation (m)
* dir   — D8 flow direction
* upa   — upstream drainage area (km²)
* wth   — river channel width (m)

In [ ]:
def exportMERIT_Hydro():
    """
    MERIT Hydro is the recommended DEM for hydrological modelling.
    It provides pre-computed flow direction and upstream area — no need
    to derive these from a raw DEM within GEE.
    """
    merit = ee.Image('MERIT/Hydro/v1_0_1')

    elev  = merit.select('elv').rename('elevation_m')
    dir_  = merit.select('dir').rename('flow_direction_D8')
    upa   = merit.select('upa').rename('upstream_area_km2')
    wth   = merit.select('wth').rename('river_width_m')

    # Log-transform upstream area to compress dynamic range for visualisation
    upa_log = upa.add(1).log().rename('upstream_area_log')

    image = ee.Image.cat([elev, dir_, upa, upa_log, wth])

    exportImage(image, f"{ImageName}_MERIT_Hydro_flow_network",
                scale=90, folder='Alaska_TRV_Terrain')
    print("  MERIT Hydro: elevation, flow direction, upstream area, river width")


def exportArcticDEM_Hydro():
    """
    ArcticDEM Mosaic (~2 m) resampled to 10 m.
    Outputs: elevation, slope, aspect, hillshade, TPI.
    Slope and TPI are most relevant for identifying flow paths and
    topographic convergence zones feeding lakes.
    """
    dem    = ee.Image('UMN/PGC/ArcticDEM/V3/2m_mosaic').select('elevation')
    slope  = ee.Terrain.slope(dem).rename('slope_deg')
    aspect = ee.Terrain.aspect(dem).rename('aspect_deg')
    hill   = ee.Terrain.hillshade(dem).rename('hillshade')

    # TPI: local relief — negative TPI = valleys/channels, positive = ridges
    kernel = ee.Kernel.circle(radius=300, units='meters')
    tpi    = dem.subtract(dem.focal_mean(kernel=kernel)).rename('TPI')

    # Terrain Ruggedness Index (TRI): mean absolute difference to neighbours
    tri    = dem.reduceNeighborhood(
                 reducer=ee.Reducer.stdDev(),
                 kernel=ee.Kernel.square(radius=1)
             ).rename('TRI')

    image  = ee.Image.cat([dem, slope, aspect, hill, tpi, tri])
    exportImage(image, f"{ImageName}_ArcticDEM_terrain_derivatives",
                scale=10, folder='Alaska_TRV_Terrain')
    print("  ArcticDEM: elevation, slope, aspect, hillshade, TPI, TRI")


def exportSRTM_Hydro():
    """
    SRTM 30 m — widely used baseline for comparison with MERIT/ArcticDEM.
    """
    srtm   = ee.Image('USGS/SRTMGL1_003').select('elevation')
    slope  = ee.Terrain.slope(srtm).rename('slope_deg')
    aspect = ee.Terrain.aspect(srtm).rename('aspect_deg')
    kernel = ee.Kernel.circle(radius=300, units='meters')
    tpi    = srtm.subtract(srtm.focal_mean(kernel=kernel)).rename('TPI')

    image  = ee.Image.cat([srtm, slope, aspect, tpi])
    exportImage(image, f"{ImageName}_SRTM_terrain_derivatives",
                scale=30, folder='Alaska_TRV_Terrain')
    print("  SRTM: elevation, slope, aspect, TPI")

# JRC Global Surface Water (1984-2024)

The JRC dataset is the gold standard for long-term lake extent and water dynamics from Landsat. Highly relevant for tracking changes in lake area receiving flow.

Outputs (static summary products):
* occurrence        — % of time water present (1984–2021)
* change_intensity  — magnitude of water change
* seasonality       — number of months water present per year
* max_extent        — ever-water mask

Outputs (annual):
* water_class       — 1=no water, 2=seasonal, 3=permanent

In [ ]:
def exportJRC_SurfaceWater_Summary():
    """
    Export the JRC long-term summary products (static mosaics).
    These show WHERE lakes and channels are and how stable they are.
    """
    jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')

    occurrence  = jrc.select('occurrence').rename('water_occurrence_pct')
    change      = jrc.select('change_abs').rename('water_change_abs')
    seasonality = jrc.select('seasonality').rename('water_seasonality_months')
    max_extent  = jrc.select('max_extent').rename('ever_water_mask')

    image = ee.Image.cat([occurrence, change, seasonality, max_extent])
    exportImage(image, f"{ImageName}_JRC_SurfaceWater_summary",
                scale=30, folder='Alaska_TRV_SurfaceWater')
    print("  JRC GSW: occurrence, change, seasonality, max extent")


def processJRC_Annual(year):
    """
    Annual JRC surface water classification.
    water_class: 0=no data, 1=no water, 2=seasonal water, 3=permanent water
    """
    col = (ee.ImageCollection('JRC/GSW1_4/MonthlyHistory')
             .filterBounds(AOI)
             .filter(ee.Filter.calendarRange(year, year, 'year'))
             .filter(ee.Filter.calendarRange(SUMMER_START, SUMMER_END, 'month')))

    count = col.size().getInfo()
    if count == 0:
        print(f"  No JRC annual data for {year}"); return None
    print(f"  {count} JRC monthly scenes for {year}")

    # water = 2 in monthly history means water present
    water_freq = (col.map(lambda img: img.select('water').eq(2).rename('water'))
                     .mean()
                     .rename('water_frequency'))

    permanent = water_freq.gte(0.75).rename('permanent_water')
    seasonal  = water_freq.gt(0).And(water_freq.lt(0.75)).rename('seasonal_water')

    return ee.Image.cat([water_freq, permanent, seasonal]).set('year', year)

# MODIS

Snowmelt timing & surface water (2000-2024)

For hydrology, snowmelt timing drives the freshet pulse into lakes.
We compute:
* Snow disappearance DOY — first DOY where NDSI < 40 after winter
* Summer open water fraction — from MOD44W or NDSI/NDWI time series
* Peak SWE proxy — max snow cover extent before melt

In [ ]:
def processMODIS_Snowmelt(year):
    """
    Identifies snowmelt timing from MOD10A1 daily snow cover.
    Exports: peak snow cover extent (pre-melt), mean snow cover (freshet),
             snow disappearance proxy (minimum snow cover day in Jun).
    """
    # Full spring window Jan–Jul to capture melt sequence
    sd = ee.Date.fromYMD(year, 1, 1)
    ed = ee.Date.fromYMD(year, 7, 31)

    col = (ee.ImageCollection('MODIS/061/MOD10A1')
             .filterBounds(AOI)
             .filterDate(sd, ed)
             .select('NDSI_Snow_Cover')
             .map(lambda img: img.updateMask(img.lte(100))))  # remove fill values

    count = col.size().getInfo()
    if count == 0:
        print(f"  No MODIS snow data for {year}"); return None
    print(f"  {count} MODIS snow scenes for {year} (Jan–Jul)")

    # Peak snow cover (max across Jan–Apr = pre-melt maximum)
    sd_premelt = ee.Date.fromYMD(year, 1,  1)
    ed_premelt = ee.Date.fromYMD(year, 4, 30)
    peak_snow  = (col.filterDate(sd_premelt, ed_premelt)
                     .max()
                     .rename('peak_snow_cover_pct'))

    # Freshet window (May–Jun) mean — indicates active melt period
    sd_freshet = ee.Date.fromYMD(year, FRESHET_START, 1)
    ed_freshet = ee.Date.fromYMD(year, FRESHET_END,   30)
    freshet    = (col.filterDate(sd_freshet, ed_freshet)
                     .mean()
                     .rename('freshet_snow_cover_pct'))

    # Snow persistence (fraction of May–Jun days with NDSI > 40)
    persistence = (col.filterDate(sd_freshet, ed_freshet)
                      .map(lambda img: img.gt(40).rename('snow'))
                      .mean()
                      .rename('snow_persistence_freshet'))

    return ee.Image.cat([peak_snow, freshet, persistence]).set('year', year)


def processMODIS_SummerWater(year):
    """
    MODIS Terra daily surface reflectance (MOD09GA) to track
    open water dynamics through summer. MNDWI = (Green-SWIR)/(Green+SWIR).
    """
    sd = ee.Date.fromYMD(year, SUMMER_START, 1)
    ed = ee.Date.fromYMD(year, SUMMER_END,   30)

    col = (ee.ImageCollection('MODIS/061/MOD09GA')
             .filterBounds(AOI)
             .filterDate(sd, ed)
             .select(['sur_refl_b04', 'sur_refl_b06'],  # Green, SWIR1 for MODIS
                     ['Green', 'SWIR1']))

    count = col.size().getInfo()
    if count == 0:
        print(f"  No MODIS MOD09GA data for {year}"); return None
    print(f"  {count} MODIS daily surface reflectance scenes for {year}")

    # MNDWI per image then mean
    mndwi_col   = col.map(lambda img:
                           img.normalizedDifference(['Green', 'SWIR1'])
                              .rename('MNDWI'))
    mean_mndwi  = mndwi_col.mean().rename('MNDWI_mean_summer')
    water_freq  = mndwi_col.map(lambda img: img.gt(0).rename('water')).mean().rename('water_frequency_summer')

    return ee.Image.cat([mean_mndwi, water_freq]).set('year', year)

# Dataset selection

In [ ]:
# Set each flag to True or False to enable/disable that dataset.
RUN_LANDSAT   = False
RUN_SAR       = True
RUN_SENTINEL2 = True
RUN_GEDI      = False
RUN_TERRAIN   = True
RUN_JRC       = True
RUN_MODIS     = False

# Terrain sub-selection (only applies if RUN_TERRAIN = True)
RUN_MERIT     = True
RUN_ARCTICDEM = True
RUN_SRTM      = False

In [ ]:
# Confirm selection

def summariseSelection():
    datasets = {
        'Landsat (1978–2024, 30m)':              RUN_LANDSAT,
        'Sentinel-1 SAR (2017–2024, 10m)':       RUN_SAR,
        'Sentinel-2 Optical (2017–2024, 10m)':   RUN_SENTINEL2,
        'GEDI LiDAR (2019–2024, 25m)':           RUN_GEDI,
        'Terrain — MERIT Hydro (static, 90m)':   RUN_TERRAIN and RUN_MERIT,
        'Terrain — ArcticDEM (static, 10m)':     RUN_TERRAIN and RUN_ARCTICDEM,
        'Terrain — SRTM (static, 30m)':          RUN_TERRAIN and RUN_SRTM,
        'JRC Surface Water (1984–2024, 30m)':    RUN_JRC,
        'MODIS Snow & Water (2000–2024, 500m)':  RUN_MODIS,
    }
    print("\n" + "="*60)
    print("EXPORT SELECTION SUMMARY")
    print("="*60)
    for name, enabled in datasets.items():
        status = "YES" if enabled else "skip"
        print(f"  [{status:4s}]  {name}")
    print("="*60)
    enabled_count = sum(1 for v in datasets.values() if v)
    print(f"  {enabled_count} of {len(datasets)} datasets enabled.")
    print("  Edit the DATASET SELECTION block above to change.\n")

summariseSelection()


EXPORT SELECTION SUMMARY
  [skip]  Landsat (1978–2024, 30m)
  [YES ]  Sentinel-1 SAR (2017–2024, 10m)
  [YES ]  Sentinel-2 Optical (2017–2024, 10m)
  [skip]  GEDI LiDAR (2019–2024, 25m)
  [YES ]  Terrain — MERIT Hydro (static, 90m)
  [YES ]  Terrain — ArcticDEM (static, 10m)
  [skip]  Terrain — SRTM (static, 30m)
  [YES ]  JRC Surface Water (1984–2024, 30m)
  [skip]  MODIS Snow & Water (2000–2024, 500m)
  5 of 9 datasets enabled.
  Edit the DATASET SELECTION block above to change.



# Main processing loops

In [ ]:
# ---- 1. LANDSAT ----
if RUN_LANDSAT:
    print("\n" + "="*60)
    print("1/7  LANDSAT COMPOSITES  (1978–2024)")
    print("="*60)

    for year in range(LANDSAT_START, LANDSAT_END + 1):
        print(f"\n--- Landsat {year} ---")
        collection_id = findBestLandsatCollection(year)
        if collection_id is None:
            print(f"  No suitable Landsat data for {year}"); continue
        print(f"  Collection: {collection_id}")

        if collection_id == 'LANDSAT/LM03/C02/T2':
            sd = ee.Date.fromYMD(year, 8, 1)
            ed = ee.Date.fromYMD(year, 9, 30)
        else:
            sd = ee.Date.fromYMD(year, SUMMER_START, 1)
            ed = ee.Date.fromYMD(year, SUMMER_END,   30)

        if collection_id == 'LANDSAT/LE07/C02/T1_L2' and year >= 2003:
            image = processLandsat7SLCOff(year, collection_id, Cloud_Max, sd, ed)
        elif collection_id == 'LANDSAT/LM03/C02/T2':
            image = processMSS(year, Cloud_Max, sd, ed)
        elif collection_id.startswith('LANDSAT/LC'):
            image = processLandsat8to9(year, collection_id, Cloud_Max, sd, ed)
        else:
            image = processLandsat4to7(year, collection_id, Cloud_Max, sd, ed)

        exportImage(image, f"{ImageName}_LS_Comp_{year}",
                    scale=30, folder='Alaska_TRV_Landsat')
        time.sleep(3)
else:
    print("\n[SKIP] Landsat — RUN_LANDSAT = False")

# ---- 2. SENTINEL-1 SAR  (two seasonal windows) ----
if RUN_SAR:
    print("\n" + "="*60)
    print("2/7  SENTINEL-1 SAR — INUNDATION  (2017–2024)")
    print("="*60)

    for year in range(SAR_START, SAR_END + 1):
        print(f"\n--- SAR {year} ---")

        sd_s, ed_s = makeDateRange(year, SUMMER_START, SUMMER_END)
        img_s = processSentinel1_Hydro(year, sd_s, ed_s, 'summer')
        exportImage(img_s, f"{ImageName}_S1_SAR_{year}_summer",
                    scale=10, folder='Alaska_TRV_SAR')
        time.sleep(3)
else:
    print("\n[SKIP] Sentinel-1 SAR — RUN_SAR = False")

# ---- 3. SENTINEL-2 WATER INDICES  (two seasonal windows) ----
if RUN_SENTINEL2:
    print("\n" + "="*60)
    print("3/7  SENTINEL-2 WATER INDICES  (2017–2024)")
    print("="*60)

    for year in range(S2_START, S2_END + 1):
        print(f"\n--- S2 {year} ---")

        sd_s, ed_s = makeDateRange(year, SUMMER_START, SUMMER_END)
        img_s = processSentinel2_Hydro(year, sd_s, ed_s, 'summer')
        exportImage(img_s, f"{ImageName}_S2_{year}_summer",
                    scale=10, folder='Alaska_TRV_S2')
        time.sleep(3)
else:
    print("\n[SKIP] Sentinel-2 — RUN_SENTINEL2 = False")

# ---- 4. GEDI GROUND ELEVATION ----
if RUN_GEDI:
    print("\n" + "="*60)
    print("4/7  GEDI GROUND ELEVATION  (2019–2024)")
    print("="*60)

    for year in range(GEDI_START, GEDI_END + 1):
        print(f"\n--- GEDI {year} ---")
        sd, ed = makeDateRange(year, SUMMER_START, SUMMER_END)
        image  = processGEDI_Hydro(year, sd, ed)
        exportImage(image, f"{ImageName}_GEDI_{year}",
                    scale=25, folder='Alaska_TRV_GEDI')
        time.sleep(3)
else:
    print("\n[SKIP] GEDI — RUN_GEDI = False")

# ---- 5. TERRAIN DERIVATIVES ----
if RUN_TERRAIN:
    print("\n" + "="*60)
    print("5/7  HYDROLOGICAL TERRAIN PRODUCTS  (static, export once)")
    print("="*60)

    if RUN_MERIT:
        exportMERIT_Hydro()
        time.sleep(3)
    else:
        print("  [SKIP] MERIT Hydro — RUN_MERIT = False")

    if RUN_ARCTICDEM:
        exportArcticDEM_Hydro()
        time.sleep(3)
    else:
        print("  [SKIP] ArcticDEM — RUN_ARCTICDEM = False")

    if RUN_SRTM:
        exportSRTM_Hydro()
        time.sleep(3)
    else:
        print("  [SKIP] SRTM — RUN_SRTM = False")
else:
    print("\n[SKIP] All terrain products — RUN_TERRAIN = False")

# ---- 6. JRC GLOBAL SURFACE WATER ----
if RUN_JRC:
    print("\n" + "="*60)
    print("6/7  JRC GLOBAL SURFACE WATER  (1984–2024)")
    print("="*60)

    exportJRC_SurfaceWater_Summary()   # single summary mosaic, no annual stack
    time.sleep(3)

else:
    print("\n[SKIP] JRC Surface Water — RUN_JRC = False")

# ---- 7. MODIS SNOWMELT & SUMMER WATER ----
if RUN_MODIS:
    print("\n" + "="*60)
    print("7/7  MODIS SNOWMELT TIMING & SUMMER WATER  (2000–2024)")
    print("="*60)

    for year in range(MODIS_START, MODIS_END + 1):
        print(f"\n--- MODIS {year} ---")

        img_snow = processMODIS_Snowmelt(year)
        exportImage(img_snow, f"{ImageName}_MODIS_Snowmelt_{year}",
                    scale=500, folder='Alaska_TRV_MODIS')
        time.sleep(2)

        img_water = processMODIS_SummerWater(year)
        exportImage(img_water, f"{ImageName}_MODIS_SummerWater_{year}",
                    scale=500, folder='Alaska_TRV_MODIS')
        time.sleep(2)
else:
    print("\n[SKIP] MODIS — RUN_MODIS = False")

print("\n" + "="*60)
print("All selected export tasks submitted!")
print("="*60)


[SKIP] Landsat — RUN_LANDSAT = False

2/7  SENTINEL-1 SAR — INUNDATION  (2017–2024)

--- SAR 2022 ---
  62 Sentinel-1 scenes for 2022 (summer)
  [EXPORT STARTED] Alaska_TRV_Basin_S1_SAR_2022_summer  (scale=10m, folder=Alaska_TRV_SAR)

--- SAR 2023 ---
  79 Sentinel-1 scenes for 2023 (summer)
  [EXPORT STARTED] Alaska_TRV_Basin_S1_SAR_2023_summer  (scale=10m, folder=Alaska_TRV_SAR)

--- SAR 2024 ---
  79 Sentinel-1 scenes for 2024 (summer)
  [EXPORT STARTED] Alaska_TRV_Basin_S1_SAR_2024_summer  (scale=10m, folder=Alaska_TRV_SAR)

3/7  SENTINEL-2 WATER INDICES  (2017–2024)

--- S2 2022 ---
  120 Sentinel-2 scenes for 2022 (summer)
  [EXPORT STARTED] Alaska_TRV_Basin_S2_2022_summer  (scale=10m, folder=Alaska_TRV_S2)

--- S2 2023 ---
  135 Sentinel-2 scenes for 2023 (summer)
  [EXPORT STARTED] Alaska_TRV_Basin_S2_2023_summer  (scale=10m, folder=Alaska_TRV_S2)

--- S2 2024 ---
  96 Sentinel-2 scenes for 2024 (summer)
  [EXPORT STARTED] Alaska_TRV_Basin_S2_2024_summer  (scale=10m, folder=Al

/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for UMN/PGC/ArcticDEM/V3/2m_mosaic! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/UMN_PGC_ArcticDEM_V3_2m_mosaic

  warnings.warn(warning, category=DeprecationWarning)


  [EXPORT STARTED] Alaska_TRV_Basin_ArcticDEM_terrain_derivatives  (scale=10m, folder=Alaska_TRV_Terrain)
  ArcticDEM: elevation, slope, aspect, hillshade, TPI, TRI
  [SKIP] SRTM — RUN_SRTM = False

6/7  JRC GLOBAL SURFACE WATER  (1984–2024)
  [EXPORT STARTED] Alaska_TRV_Basin_JRC_SurfaceWater_summary  (scale=30m, folder=Alaska_TRV_SurfaceWater)
  JRC GSW: occurrence, change, seasonality, max extent

[SKIP] MODIS — RUN_MODIS = False

All selected export tasks submitted!


# Task manager

In [ ]:
def checkRecentTasks(hours_ago=72):
    import datetime
    tasks        = ee.batch.Task.list()
    current_time = datetime.datetime.now()
    running = completed = failed = ready = 0
    filtered_tasks = []

    for task in tasks:
        status = task.status()
        if 'creation_timestamp_ms' in status:
            creation_time = datetime.datetime.fromtimestamp(
                int(status['creation_timestamp_ms']) / 1000
            )
            if (current_time - creation_time).total_seconds() <= hours_ago * 3600:
                filtered_tasks.append((status, creation_time))
                state = status['state']
                if state == 'RUNNING':    running   += 1
                elif state == 'COMPLETED': completed += 1
                elif state in ['FAILED', 'CANCELLED']: failed += 1
                elif state == 'READY':    ready     += 1

    filtered_tasks.sort(key=lambda x: x[1], reverse=True)
    print(f"\nTask Status (last {hours_ago} hours):")
    for status, creation_time in filtered_tasks:
        state = status['state']
        print(f"  {creation_time.strftime('%Y-%m-%d %H:%M')}  "
              f"{status['description']}  [{state}]")
        if state in ['FAILED', 'CANCELLED'] and 'error_message' in status:
            print(f"    Error: {status['error_message']}")

    print(f"\nSummary: {running} running | {ready} ready | "
          f"{completed} completed | {failed} failed")

checkRecentTasks(72)


Task Status (last 72 hours):
  2026-02-21 08:47  Alaska_TRV_Basin_JRC_SurfaceWater_summary  [COMPLETED]
  2026-02-21 08:47  Alaska_TRV_Basin_ArcticDEM_terrain_derivatives  [COMPLETED]
  2026-02-21 08:47  Alaska_TRV_Basin_MERIT_Hydro_flow_network  [COMPLETED]
  2026-02-21 08:47  Alaska_TRV_Basin_S2_2024_summer  [COMPLETED]
  2026-02-21 08:47  Alaska_TRV_Basin_S2_2023_summer  [COMPLETED]
  2026-02-21 08:47  Alaska_TRV_Basin_S2_2022_summer  [COMPLETED]
  2026-02-21 08:47  Alaska_TRV_Basin_S1_SAR_2024_summer  [COMPLETED]
  2026-02-21 08:47  Alaska_TRV_Basin_S1_SAR_2023_summer  [COMPLETED]
  2026-02-21 08:47  Alaska_TRV_Basin_S1_SAR_2022_summer  [COMPLETED]
  2026-02-21 00:52  Alaska_TRV_Basin_MODIS_SummerWater_2024  [COMPLETED]
  2026-02-21 00:52  Alaska_TRV_Basin_MODIS_Snowmelt_2024  [COMPLETED]
  2026-02-21 00:52  Alaska_TRV_Basin_MODIS_SummerWater_2023  [COMPLETED]
  2026-02-21 00:52  Alaska_TRV_Basin_MODIS_Snowmelt_2023  [COMPLETED]
  2026-02-21 00:52  Alaska_TRV_Basin_MODIS_SummerWat